In [5]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [6]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
    ]
})

print(response)

{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='0583b214-0230-418c-8ef6-6ac0a5c450d5'), HumanMessage(content='你好，我是虎哥.', additional_kwargs={}, response_metadata={}, id='3892f707-2b85-4286-b102-95acea84f5cf'), AIMessage(content='你好，虎哥，很高兴认识你.', additional_kwargs={}, response_metadata={}, id='cfaea915-df95-4edc-aa28-47e180885a7b', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='51548da2-d277-4835-9593-0aeeabba5ad5'), AIMessage(content='北京今天的天气信息如下：', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 309, 'total_tokens': 352, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 181}, 'model_provider': 'deepseek', 'model_n

In [7]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎哥.
================================== Ai Message ==================================

你好，虎哥，很高兴认识你.
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================

北京今天的天气信息如下：
Tool Calls:
  get_weather (call_00_TizWszJ8lV9Q6fTTEi2x1475)
 Call ID: call_00_TizWszJ8lV9Q6fTTEi2x1475
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

Current weather in 北京 is sunny
================================== Ai Message ==================================

虎哥，北京今天天气晴朗。

需要我帮你查其他城市，或者了解更多天气细节吗？


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [8]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [9]:
# 创建Agent
agent = create_agent(model=model)

In [10]:
# 准备多模态消息
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [11]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

这是一张充满温馨和治愈感的照片，捕捉了一个年轻女子和她的宠物狗在海滩上互动的瞬间。以下是详细的画面描述：

**1. 主体与动作：**
*   **女子：** 画面右侧坐着一位长发年轻女子。她身穿黑白格纹的长袖衬衫和深色长裤（裤脚卷起），赤脚坐在沙滩上。她面带灿烂的笑容，侧身面向左侧，看起来非常开心。
*   **狗：** 画面左侧是一只浅金色的大型犬（看起来像拉布拉多或金毛寻回犬）。它乖巧地坐在沙地上，身上戴着带有彩色花纹的胸背带，红色的牵引绳散落在它身后的沙地上。
*   **互动：** 这是一个经典的“握手”场景。狗狗抬起它的右前爪，正放在女子的手掌中。女子的左手握着狗的爪子，右手似乎拿着小零食，正准备奖励它。

**2. 环境与氛围：**
*   **背景：** 背景是广阔的大海和沙滩。可以看到远处平静的海平面和微微卷起的海浪。
*   **光线：** 此时正值日落（或者是日出）时分，也就是所谓的“黄金时刻”。强烈的阳光从画面右侧照射过来，形成了美丽的逆光效果，给女子的头发和身体轮廓镀上了一层金色的光晕。
*   **整体感觉：** 柔和的暖色调光线、细腻的沙滩纹理以及人与动物之间亲密的互动，共同营造出一种宁静、快乐且充满爱意的氛围。

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [ ]:
!uv add ipywidgets

然后，我们创建一个上传组件，用于模拟图片上传。


In [12]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [14]:
print(uploader.value)

({'name': '屏幕截图 2024-11-28 173041.png', 'type': 'image/png', 'size': 44984, 'content': <memory at 0x0000018D1FE6D780>, 'last_modified': datetime.datetime(2024, 11, 28, 9, 30, 41, 802000, tzinfo=datetime.timezone.utc)},)


In [15]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [16]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

这张图片里其实**并没有展示任何城市**。

图片背景是纯白色的，主体是一个动漫角色的侧脸特写。

不过，既然你问到了城市，我猜你可能是在问这个角色所属的故事背景：

1.  **角色身份**：这个银白色刺猬头、穿着黑色高领衣服的角色，看起来非常像动漫《咒术回战》（Jujutsu Kaisen）中的人气角色**五条悟**（Gojo Satoru）。
2.  **相关城市**：如果是指这部作品的主要发生地，那就是**日本东京**。故事主要围绕着“东京都立咒术高等专门学校”展开，大部分剧情都发生在东京及其周边地区（比如涩谷事变等著名篇章）。

如果你是想让我介绍某个具体的城市风景，可能发错图片啦！你可以重新发一张有街道或建筑的照片给我。